# Critical analysis: data pipeline, ViT system, and eye-tracking alignment

**Purpose:** Onboard engineers to the Surgical Gestures project—what we are building, how JIGSAWS data becomes tensors, how the ViT stack works, and **how eye-tracking-derived structure enters training** (not as per-frame gaze channels).

**Related docs:** `current_status.md`, `full_scope_plan.md`.

**Prerequisites:** Run this notebook from the **repository root** (so `Gestures/`, `src/`, and `Eye/` resolve). Requires `torch`, `timm`, `opencv-python`, `pyyaml`, `scipy`, and the JIGSAWS + eye assets on disk.

## Executive summary

1. **Goal:** A **brain-informed** vision pipeline for **surgical gesture recognition**, **skill classification**, and **per-timestep kinematics** from video, with a path toward dVRK execution (see project docs for full scope).
2. **Core stack:** Video frames → **ViT** encoder → **temporal transformer** → **kinematics decoder** + gesture/skill heads.
3. **Eye tracking in training:** Raw gaze CSVs are **not** fed frame-by-frame into the model. Eye data were used **offline** to build a fixed **3×3 representational dissimilarity matrix (RDM)** over the three JIGSAWS tasks. During training, a **task-centroid RDM** is computed from ViT embeddings and aligned to that target via **RSA loss** (`L_brain`), jointly with kinematics and classification losses.
4. **Evaluation:** Leave-one-user-out (LOUO) folds in `data/splits/`.

The diagram below matches the implementation in `src/training/train_vit_system.py` and `src/modules/brain_rdm.py`.

```mermaid
flowchart LR
  subgraph vision [Vision path]
    V[Video frames] --> ViT[ViTFrameEncoder]
    ViT --> Temp[TemporalAggregator]
    Temp --> Kin[KinematicsModule]
    Kin --> Khat["k_hat kinematics"]
    Kin --> G["gesture logits"]
    Kin --> S["skill logits"]
  end
  subgraph brain [Brain alignment training only]
    ViT --> Emb["embeddings early or mid or late"]
    Emb --> Mrdm["compute_task_centroid_rdm"]
    TL[task_label batch] --> Mrdm
    EyeFile["target_rdm_3x3.npy from eye exploration"] --> Trdm["target RDM"]
    Mrdm --> RSA["eye_rsa_loss"]
    Trdm --> RSA
  end
  RSA --> Ltot["L_total via compute_total_loss"]
  Khat --> Ltot
  G --> Ltot
  S --> Ltot
```

## 1. Environment: paths and imports

Add `src/` to `sys.path` so `models`, `data`, and `modules` import like the training script.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import torch
import yaml

# Repository root = cwd (run notebook from project root)
PROJECT_ROOT = Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data.jigsaws_multitask_dataset import JIGSAWSMultiTaskDataset, TASK_TO_LABEL
from modules.brain_rdm import (
    compute_task_centroid_rdm,
    eye_rsa_loss,
    load_eye_rdm,
)
from training.train_vit_system import EEGInformedViTModel
from torch.utils.data import DataLoader, Subset

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TASK_TO_LABEL:", TASK_TO_LABEL)

PROJECT_ROOT: /Users/michaelhaidar/Documents/Vanderbilt/Fall_25/Surgical Robotics/Surgical_Gestures
TASK_TO_LABEL: {'Suturing': 0, 'Needle_Passing': 1, 'Knot_Tying': 2}


## 2. Config inspection: baseline vs brain-eye

`brain_eye.yaml` enables `brain_mode: eye`, points to the precomputed eye RDM, and sets `loss_weights.brain`. The baseline turns brain alignment off.

In [2]:
def load_yaml(name: str) -> dict:
    path = PROJECT_ROOT / "src" / "configs" / name
    with open(path, "r") as f:
        return yaml.safe_load(f)

cfg_brain = load_yaml("brain_eye.yaml")
cfg_base = load_yaml("baseline.yaml")

for label, cfg in [("brain_eye.yaml", cfg_brain), ("baseline.yaml", cfg_base)]:
    print(f"\n=== {label} ===")
    print("  brain_mode:", cfg.get("brain_mode"))
    print("  eye_rdm_path:", cfg.get("eye_rdm_path"))
    print("  brain_layer:", cfg.get("brain_layer"))
    print("  loss_weights:", cfg.get("loss_weights"))


=== brain_eye.yaml ===
  brain_mode: eye
  eye_rdm_path: Eye/Exploration/target_rdm_3x3.npy
  brain_layer: mid
  loss_weights: {'kin': 1.0, 'gesture': 1.0, 'skill': 0.5, 'brain': 0.01, 'control': 0.01}

=== baseline.yaml ===
  brain_mode: none
  eye_rdm_path: None
  brain_layer: None
  loss_weights: {'kin': 1.0, 'gesture': 1.0, 'skill': 0.5, 'brain': 0.0, 'control': 0.01}


## 3. Data processing pipeline (JIGSAWS)

**Source layout:** `Gestures/<Task>/` contains `video/`, `kinematics/AllGestures/`, `transcriptions/`, and `meta_file_<Task>.txt`.

**How samples are built** (`src/data/jigsaws_vit_dataset.py`):

1. Each line in a transcription file defines a **gesture segment**: start frame, end frame, gesture id (e.g. `G5`).
2. For each segment, the dataset loads the matching **video** clip, **resamples kinematics** to frame indices via `SyncManager` (`src/data/sync_manager.py`), and extracts **one arm** (default PSM2 → 19-D).
3. **Temporal windows** (e.g. gesture task: 10 frames, stride 5) crop the tensor that the ViT sees.

**Multi-task training** (`src/data/jigsaws_multitask_dataset.py`): concatenates Suturing, Needle Passing, and Knot Tying with LOUO splits and adds **`task_label` ∈ {0,1,2}** so the training loop can form **per-task centroid embeddings** for the 3×3 RDM.

Below: parse **one** transcription line and optionally peek at **eye-tracking** columns from a sample CSV.

In [3]:
trans_dir = PROJECT_ROOT / "Gestures" / "Knot_Tying" / "transcriptions"
trans_files = sorted(trans_dir.glob("*.txt"))
assert trans_files, f"No transcriptions under {trans_dir}"

sample_txt = trans_files[0]
trial_id = sample_txt.stem
with open(sample_txt) as f:
    first_line = f.readline().strip()

parts = first_line.split()
start_f, end_f, gesture = int(parts[0]), int(parts[1]), parts[2]
print("Example transcription file:", sample_txt.name)
print("  trial_id:", trial_id)
print("  first line (start_frame, end_frame, gesture):", start_f, end_f, gesture)

# Eye CSV: prefer Eye/EYE; fall back to root copy if present
eye_candidates = [
    PROJECT_ROOT / "Eye" / "EYE" / "10_11_1.csv",
    PROJECT_ROOT / "10_11_1.csv",
]
eye_path = next((p for p in eye_candidates if p.exists()), None)
if eye_path is not None:
    import csv

    rows = []
    with open(eye_path, newline="") as f:
        r = csv.reader(f)
        for i, row in enumerate(r):
            if i >= 3:
                break
            rows.append([row[0], row[1], row[17], row[18], row[19]])
    print("\nEye CSV sample (first 3 rows): gaze_x, gaze_y, pupil_left, pupil_right, eye_movement_type")
    print("File:", eye_path)
    for row in rows:
        print(" ", row)
    print("eye_movement_type: 1=Fixation, 2=Saccade, 0/3=Noise (per project docs)")
else:
    print("No sample eye CSV found at Eye/EYE/ or repo root — see current_status.md for column layout.")

Example transcription file: Knot_Tying_B001.txt
  trial_id: Knot_Tying_B001
  first line (start_frame, end_frame, gesture): 45 85 G12

Eye CSV sample (first 3 rows): gaze_x, gaze_y, pupil_left, pupil_right, eye_movement_type
File: /Users/michaelhaidar/Documents/Vanderbilt/Fall_25/Surgical Robotics/Surgical_Gestures/Eye/EYE/10_11_1.csv
  ['826', '1056', '4.36', '4.49', '2']
  ['791', '1007', '4.53', '4.52', '2']
  ['774', '972', '4.63', 'NaN', '1']
eye_movement_type: 1=Fixation, 2=Saccade, 0/3=Noise (per project docs)


## 4. Sample batch (small subset for speed)

We use `JIGSAWSMultiTaskDataset` with **fold_1** so each sample carries **`task_label`**, required for task-centroid RSA. Only the **first few indices** are used so the notebook stays fast.

In [4]:
SAMPLE_N = 8
BATCH_SIZE = min(4, SAMPLE_N)

full_ds = JIGSAWSMultiTaskDataset(
    str(PROJECT_ROOT),
    split_name="fold_1",
    mode="train",
    task_type="gesture",
    use_rgb=True,
    use_flow=False,
    arm="PSM2",
)
n = min(SAMPLE_N, len(full_ds))
subset = Subset(full_ds, range(n))
loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch = next(iter(loader))
print("Dataset size (subset):", n)
print("Batch keys:", sorted(batch.keys()))
print("rgb:", batch["rgb"].shape)
print("kinematics:", batch["kinematics"].shape)
print("task_label:", batch["task_label"])

Dataset size (subset): 8
Batch keys: ['gesture_label', 'kinematics', 'rgb', 'skill_label', 'surgeon_id', 'task_label', 'trial_id']
rgb: torch.Size([4, 10, 3, 224, 224])
kinematics: torch.Size([4, 10, 19])
task_label: tensor([0, 0, 0, 0])


## 5. ViT system: one forward pass with embeddings

`EEGInformedViTModel` matches `src/training/train_vit_system.py`: ViT → temporal stack → kinematics module. Set `return_embeddings=True` to mirror the **eye** training path.

In [5]:
device = torch.device("cpu")
config = dict(cfg_brain)
model = EEGInformedViTModel(config).to(device)
model.eval()

rgb = batch["rgb"].to(device)
kin = batch["kinematics"].to(device)

with torch.no_grad():
    out = model(
        rgb,
        target_kinematics=kin,
        teacher_forcing_prob=1.0,
        return_embeddings=True,
    )

brain_layer = config.get("brain_layer", "mid")
emb_dict = out["embeddings"]
print("Output keys:", sorted(out.keys()))
print("kinematics pred:", out["kinematics"].shape)
print("gesture_logits:", out["gesture_logits"].shape)
print("Embedding keys:", list(emb_dict.keys()))
print(f"embeddings[{brain_layer!r}]:", emb_dict[brain_layer].shape)

Initializing EEGInformedViTModel
Creating visual encoder: vit_small_patch16_224
  - Pretrained: True
  - Freeze until layer: 6
  - Use adapters: True
Flow encoder: disabled
Creating temporal aggregator...
  - d_model: 384, n_heads: 6, num_layers: 4
Creating kinematics module...
  - d_kin_input: 19, d_kin_output: 19
  - num_gestures: 15, num_skills: 3
Creating Brain RDM module (mode: eye)...

Model Parameters:
  - Total: 46,165,541 (46.17M)
  - Trainable: 34,552,613 (34.55M)
Output keys: ['embeddings', 'gesture_logits', 'kinematics', 'memory', 'skill_logits']
kinematics pred: torch.Size([4, 10, 19])
gesture_logits: torch.Size([4, 15])
Embedding keys: ['early', 'mid', 'late']
embeddings['mid']: torch.Size([4, 10, 384])


## 6. Eye-tracking alignment: task-centroid RDM vs target RDM

**Training logic** (same as `train_vit_system.train_epoch` when `brain_mode == 'eye'`):

1. Take ViT embeddings for the configured layer, shape `(B, T, D)`.
2. Flatten to `(B*T, D)` and broadcast **`task_label`** to each frame.
3. **`compute_task_centroid_rdm`**: mean embedding per task (0=Suturing, 1=Needle Passing, 2=Knot Tying), then pairwise Euclidean distances → **3×3** matrix.
4. Load **`target_rdm_3x3.npy`** (built from eye-tracking exploration — fixed matrix, not per-batch gaze).
5. **`eye_rsa_loss`**: `1 - PearsonCorr` between upper triangles (differentiable in training).

If the numpy file is missing, we use a **synthetic** 3×3 matrix so the cell still runs.

In [6]:
rdm_path = PROJECT_ROOT / cfg_brain.get("eye_rdm_path", "Eye/Exploration/target_rdm_3x3.npy")
if not rdm_path.is_absolute():
    rdm_path = PROJECT_ROOT / rdm_path

if rdm_path.exists():
    target_rdm = load_eye_rdm(str(rdm_path))
    print("Loaded target RDM from:", rdm_path)
else:
    # Demo-only fallback if exploration artifact is absent
    target_rdm = torch.tensor(
        [[0.0, 0.4, 0.6], [0.4, 0.0, 0.5], [0.6, 0.5, 0.0]], dtype=torch.float32
    )
    print("WARNING: using synthetic target RDM (file missing):", rdm_path)

print("target_rdm (3x3):\n", target_rdm.numpy())

emb = out["embeddings"][brain_layer]
B, T, D = emb.shape
flat = emb.view(B * T, D).detach()
task_labels = batch["task_label"]
task_exp = task_labels.unsqueeze(1).expand(-1, T).reshape(B * T)

model_rdm = compute_task_centroid_rdm(flat, task_exp)
print("model_rdm (3x3):\n", model_rdm.numpy())

l_brain = eye_rsa_loss(model_rdm, target_rdm)
print("L_brain (eye_rsa_loss):", float(l_brain))

w = cfg_brain.get("loss_weights", {}).get("brain", 0.01)
print(f"Weighted λ_brain * L_brain (λ={w}):", float(w * l_brain))

Loaded target RDM from: /Users/michaelhaidar/Documents/Vanderbilt/Fall_25/Surgical Robotics/Surgical_Gestures/Eye/Exploration/target_rdm_3x3.npy
target_rdm (3x3):
 [[0.         1.         0.3952072 ]
 [1.         0.         0.61767286]
 [0.3952072  0.61767286 0.        ]]
model_rdm (3x3):
 [[9.9999997e-05 1.7053894e+01 1.7053894e+01]
 [1.7053894e+01 9.9999997e-05 9.9999997e-05]
 [1.7053894e+01 9.9999997e-05 9.9999997e-05]]
L_brain (eye_rsa_loss): 0.8491388559341431
Weighted λ_brain * L_brain (λ=0.01): 0.008491388522088528


## 7. Loss composition (`src/models/losses.py`)

Total loss combines kinematics, gesture CE, skill CE, **brain RSA** (when enabled), and a control regularizer. The **eye** branch is:

```python
if brain_mode == 'eye' and model_rdm is not None and eeg_rdm is not None and eye_rsa_loss is not None:
    brain_loss = eye_rsa_loss(model_rdm, eeg_rdm)
```

The snippet below prints the actual source lines from your checkout (for traceability).

In [7]:
from models import losses

losses_py = (PROJECT_ROOT / "src" / "models" / "losses.py").read_text().splitlines()
for i, line in enumerate(losses_py):
    if "Brain alignment loss" in line:
        print("Excerpt from losses.py (brain + total):")
        for j in range(i, min(i + 28, len(losses_py))):
            print(losses_py[j])
        break

_, comp = losses.compute_total_loss(
    out["kinematics"],
    batch["kinematics"].to(device),
    out["gesture_logits"],
    batch["gesture_label"].to(device),
    out["skill_logits"],
    batch["skill_label"].to(device),
    model_rdm=model_rdm,
    eeg_rdm=target_rdm.to(device),
    brain_mode="eye",
    loss_weights=cfg_brain.get("loss_weights"),
    kinematics_format="19d",
)
print("\nExample component losses (same batch):")
for k in sorted(comp.keys()):
    v = comp[k]
    print(f"  {k}: {float(v):.6f}")

Excerpt from losses.py (brain + total):
    # Brain alignment loss
    brain_loss = torch.tensor(0.0, device=pred_kinematics.device)
    if brain_mode == 'eye' and model_rdm is not None and eeg_rdm is not None and eye_rsa_loss is not None:
        # Eye-tracking task-centroid RSA (differentiable)
        brain_loss = eye_rsa_loss(model_rdm, eeg_rdm)
        component_losses['brain_rsa'] = brain_loss
    elif brain_mode == 'rsa' and model_rdm is not None and eeg_rdm is not None:
        brain_loss = rsa_loss(model_rdm, eeg_rdm)
        component_losses['brain_rsa'] = brain_loss
    elif brain_mode == 'encoding' and model_features is not None and eeg_patterns is not None:
        brain_loss = encoding_loss(model_features, eeg_patterns)
        component_losses['brain_encoding'] = brain_loss
    
    # Control regularizer
    control_reg = control_regularizer(pred_kinematics)
    component_losses['control'] = control_reg
    
    # Total loss
    total_loss = (
        loss_weights['kin']

## 8. Critical analysis (engineering takeaways)

**Strengths**

- **Clear separation of concerns:** `data/` (JIGSAWS loading + splits), `models/` (ViT, temporal, kinematics heads), `modules/brain_rdm.py` (RDM utilities), `training/train_vit_system.py` (loop).
- **Eye alignment is explicit:** A fixed **3×3** target RDM and **differentiable** `eye_rsa_loss` make the inductive bias inspectable (compare matrices side-by-side).
- **Multi-task hook:** `task_label` is the minimal extra label needed to relate **video embeddings** to **task-level** geometry.

**Limitations / caveats**

- **Not gaze fusion:** Eye CSV streams are **not** synchronized with video frames in the dataloader; alignment is **representational** (RSA), not supervised dense gaze prediction.
- **Task-level RDM only:** A single 3×3 matrix cannot encode fine-grained trial or surgeon variability in eye behavior—only coarse **between-task** structure carried into `target_rdm_3x3.npy`.
- **Coverage:** Centroid RDM needs batches that include multiple tasks; extremely small batches can make centroids noisy (training uses larger batches across tasks).
- **Dependencies:** The ViT path requires `timm`, CUDA/MPS optional; data loading requires **OpenCV** for video IO.

**What “understood the system” looks like**

- You can trace one batch from **filesystem paths → tensors → loss terms**, and explain why **`brain_mode: eye` needs multi-task data + `task_label`**, while baseline single-task training does not compute `L_brain` the same way.